# Etapa 2 — Propagación del humo y accesibilidad

**Objetivo:** comprobar `Entorno.actualizar_humo()` y `Entorno.es_transitable()` sin utilizar BFS ni agentes.

Primero se realizan **15 actualizaciones** del humo y se lo representa gráficamente. Después se comprueba que se propague desde el foco a una celda vecina, que permanezca entre 0 y 1 y que las paredes no acumulen humo.

Finalmente, se asignan concentraciones **manualmente** a algunas celdas para comprobar las reglas de accesibilidad sin depender de la difusión.

**Antes de ejecutar:** completar `Entorno.actualizar_humo()`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from modelo import Configuracion, Entorno
from visualizacion import mostrar_humo

configuracion = Configuracion(usar_humo=True, posicion_foco="izquierda", umbral_bloqueo=0.40)
entorno = Entorno(configuracion)

# La matriz comienza sin humo: actualizarla 15 veces.
for paso in range(15):
    entorno.actualizar_humo()

# Visualizar cómo se distribuyó el humo.
figura = mostrar_humo(entorno)
plt.show()

## Prueba 1. Propagación y límites

El foco incluye la celda `(4, 7)`. Su vecina `(5, 7)` **no es una fuente**: si después de 15 pasos contiene humo, se comprueba que hubo propagación. También se verifican los límites de concentración y que las paredes permanezcan sin humo.

In [ ]:
# Las celdas que emiten humo deben tener concentración positiva.
assert all(entorno.humo[y, x] > 0 for x, y in entorno.fuentes)

# La celda (5, 7) está a la derecha de la fuente (4, 7).
# En matrices NumPy las posiciones se consultan como [y, x].
assert entorno.humo[7, 5] > 0

# Las concentraciones se normalizan entre 0 y 1.
assert np.min(entorno.humo) >= 0
assert np.max(entorno.humo) <= 1

# Las paredes no pueden acumular humo.
assert np.all(entorno.humo[entorno.paredes] == 0)
print("Prueba 1: propagación y límites correctos.")

## Prueba 2. Umbral de accesibilidad

Se modifica **artificialmente** el humo en una celda libre: debajo de 0,40 se puede ingresar y al alcanzar 0,40 deja de ser transitable.

También se fija el humo de una **salida en 0,80** para probar explícitamente que continúa accesible aunque supere el umbral, según la hipótesis del modelo. Las paredes y el foco siempre deben permanecer bloqueados.

In [ ]:
# Celda libre por debajo del umbral: se permite entrar.
x, y = 12, 12
entorno.humo[y, x] = configuracion.umbral_bloqueo - 0.01
assert entorno.es_transitable(x, y)

# Al alcanzar el umbral, deja de permitirse el ingreso.
entorno.humo[y, x] = configuracion.umbral_bloqueo
assert not entorno.es_transitable(x, y)

# Se asigna humo alto directamente a una salida: debe seguir accesible.
x_salida, y_salida = entorno.salidas[0]
entorno.humo[y_salida, x_salida] = 0.80
assert entorno.es_transitable(x_salida, y_salida)

# Las paredes y las fuentes del incendio nunca son transitables.
assert not entorno.es_transitable(0, 0)
assert all(not entorno.es_transitable(x, y) for x, y in entorno.fuentes)

print("Prueba 2: accesibilidad correcta.")

**Para discutir:** una persona ya situada sobre una celda bloqueada puede abandonarla; la accesibilidad se aplica a **nuevos ingresos**, no al lugar que ocupa el agente.